In [2]:
from pathlib import Path
import geopandas as gpd
import shutil
from RA2CE_Run_functions_module import load_region_shapefile, load_networks, filter_network_by_region, create_segments_simple, move_output_file
from ra2ce.network.network_config_data.network_config_data import NetworkConfigData, NetworkSection, HazardSection
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.analysis.damages.damages import AnalysisSectionDamages
from ra2ce.analysis.analysis_config_data.enums.analysis_damages_enum import AnalysisDamagesEnum
from ra2ce.analysis.analysis_config_data.enums.event_type_enum import EventTypeEnum
from ra2ce.analysis.analysis_config_data.enums.damage_curve_enum import DamageCurveEnum
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisConfigData
from ra2ce.ra2ce_handler import Ra2ceHandler


c:\Users\gunaratn\AppData\Local\miniforge3\envs\ra2ce_env_brs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
region_list = ["Overijsselse Vecht"
               ]

In [ ]:

region_shapefile = Path(r"P:\bovenregionale-stresstest-hwn\Data\Hazard_maps\Gebiedsindeling obv waterschappen EXPORT 4dec24.gpkg")
HWN_network = Path(r"P:\bovenregionale-stresstest-hwn\Data\Shapes netwerkschakels\Shapes netwerkschakels\HWN_netwerkindeling.shp")
NWB_network = Path(r"P:\bovenregionale-stresstest-hwn\Data\Road_data\Rijkswegen_uit_nwb\rijkswegen.shp")

In [4]:
for region in region_list:
    print("creating inputs for RA2CE run")
    region_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}")
    input_dir = region_dir.joinpath("Inputs")
    output_path = region_dir.joinpath("Outputs")
    static_path = input_dir.joinpath("static")
    overlay_path = static_path.joinpath("output_graph")
    network_path = static_path.joinpath("network")
    hazard_path = static_path.joinpath("hazard")

    all_maps = list(hazard_path.glob("*.tif"))
    print(all_maps)
    
    # So the flood depth map would always be processed first
    hazard_map = []

    # Add the 'merge' map first (required)
    merge_maps = [Path(file) for file in all_maps if "max_wd_merge" in file.name.lower()]
    if not merge_maps:
        raise FileNotFoundError("No hazard map with 'merge' in the filename was found.")
    hazard_map.append(merge_maps[0])  # Assuming only one merge map

    # Add the 'duur' map if it exists (optional)
    duur_maps = [Path(file) for file in all_maps if "duur" in file.name.lower()]
    if duur_maps:
        hazard_map.append(duur_maps[0]) 

    root_dir = input_dir
    network_section = NetworkSection(
        source=SourceEnum.SHAPEFILE,
        primary_file=[network_path.joinpath(f"{region}_segmented_network.gpkg")],
        file_id="REF_ID",
        link_type_column="highway",
        save_gpkg=True
    )

    hazard = HazardSection(
        hazard_map=hazard_map,
        aggregate_wl=AggregateWlEnum.MEAN,
        hazard_crs="EPSG:28992"
    )

    network_config_data = NetworkConfigData(
        root_path=root_dir,
        static_path=static_path,
        output_path=output_path,
        network=network_section,
        hazard=hazard
    )
    
    section_damage = [AnalysisSectionDamages(
    name='ML_damage',
    analysis=AnalysisDamagesEnum.DAMAGES,
    event_type=EventTypeEnum.EVENT,
    damage_curve=DamageCurveEnum.MAN,
    save_gpkg=True,
    save_csv=True,
    )]
    
    analysis_config_data = AnalysisConfigData(analyses=section_damage, root_path=root_dir, output_path=output_path)
    analysis_config_data.input_path = root_dir.joinpath("input_data")

    print("Creating RA2CE handler and running analysis")
    handler = Ra2ceHandler.from_config(network_config_data, analysis_config_data)
    handler.configure()
    handler.run_analysis()

    move_output_file(overlay_path, output_path, "base_network_hazard.gpkg")

creating inputs for RA2CE run
[WindowsPath('P:/bovenregionale-stresstest-hwn/Analysis/Overijsselse Vecht/Inputs/static/hazard/OverijsselseVecht__max_wd_merge_clipNL.tif'), WindowsPath('P:/bovenregionale-stresstest-hwn/Analysis/Overijsselse Vecht/Inputs/static/hazard/OverijsselseVecht_duur_merge_clipNL.tif')]
Creating RA2CE handler and running analysis
File base_network_hazard.gpkg not found in P:\bovenregionale-stresstest-hwn\Analysis\Overijsselse Vecht\Inputs\static\output_graph
